# Databricks Metadata Table Extracton Automation

In [0]:
# Install required dependency (Databricks notebooks usually have this preinstalled)
# %pip install openpyxl

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font

In [0]:
# ------------------------------------------------------------
# Databricks Widgets (Environment Parameters)
# ------------------------------------------------------------
dbutils.widgets.text("environment", "")
dbutils.widgets.text("data_layer", "")

environment = dbutils.widgets.get("environment")
data_layer = dbutils.widgets.get("data_layer")

print(f"Environment: {environment}")
print(f"Data Layer: {data_layer}")

In [0]:
# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
INPUT_EXCEL_PATH = "/path/to/input.xlsx"
OUTPUT_EXCEL_PATH = "/path/to/output.xlsx"

SOURCE_INPUT_COLUMN = "Source_Table"
TARGET_OUTPUT_COLUMN = "Resolved_Databricks_Table"

CATALOG_LIST = [
    f"{environment}_core_db",
    f"{environment}_integration_db",
    f"{environment}_analytics_db",
    f"{environment}_staging_db"
]

In [0]:
# ------------------------------------------------------------
# Step 1: Read Input Excel
# ------------------------------------------------------------
input_df = pd.read_excel(INPUT_EXCEL_PATH)

# ------------------------------------------------------------
# Step 2: Load Metadata from Databricks Information Schema
# ------------------------------------------------------------
print("Loading table and view metadata from information_schema...")

tables_df = spark.sql(f"""
    SELECT table_catalog, table_schema, table_name
    FROM system.information_schema.tables
    WHERE table_catalog IN ({','.join([f"'{c}'" for c in CATALOG_LIST])})
""").toPandas()

views_df = spark.sql(f"""
    SELECT table_catalog, table_schema, table_name
    FROM system.information_schema.views
    WHERE table_catalog IN ({','.join([f"'{c}'" for c in CATALOG_LIST])})
""").toPandas()

metadata_df = pd.concat([tables_df, views_df], ignore_index=True)

# Create lookup index: FULL_NAME -> FULL_NAME
metadata_lookup = {}
for _, row in metadata_df.iterrows():
    full_path = f"{row['table_catalog']}.{row['table_schema']}.{row['table_name']}"
    metadata_lookup[full_path.upper()] = full_path

# ------------------------------------------------------------
# Step 3: Table Resolution Logic
# ------------------------------------------------------------
def resolve_table_name(source_name: str) -> str:
    if not isinstance(source_name, str) or not source_name.strip():
        return "NOT FOUND"

    source_name = source_name.strip().upper()

    # Step 1: Direct match on table name across catalogs
    for key, value in metadata_lookup.items():
        if key.endswith(f".{source_name}"):
            return value

    # Step 2: Prefix-based parsing logic (generic)
    if "_" in source_name:
        parts = source_name.split("_", 1)
        schema_hint = parts[0].lower()
        table_name = parts[1]

        for catalog in CATALOG_LIST:
            lookup_key = f"{catalog}.{schema_hint}.{table_name}".upper()
            if lookup_key in metadata_lookup:
                return metadata_lookup[lookup_key]

    # Step 3: Fallback
    return "NOT FOUND"

# ------------------------------------------------------------
# Step 4: Apply Resolution Logic
# ------------------------------------------------------------
resolved_values = []
not_found_flags = []

for value in input_df[SOURCE_INPUT_COLUMN]:
    if pd.isna(value):
        resolved_values.append("")
        not_found_flags.append(False)
        continue

    table_list = [x.strip() for x in str(value).split(",")]
    resolved_list = [resolve_table_name(t) for t in table_list]

    resolved_values.append(", ".join(resolved_list))
    not_found_flags.append(any(r == "NOT FOUND" for r in resolved_list))

input_df[TARGET_OUTPUT_COLUMN] = resolved_values

# ------------------------------------------------------------
# Step 5: Write Output Excel (Temporary)
# ------------------------------------------------------------
TEMP_EXCEL_PATH = "/path/to/temp_output.xlsx"
input_df.to_excel(TEMP_EXCEL_PATH, index=False)

# ------------------------------------------------------------
# Step 6: Highlight NOT FOUND Entries in Red
# ------------------------------------------------------------
workbook = load_workbook(TEMP_EXCEL_PATH)
worksheet = workbook.active

header_row = [cell.value for cell in worksheet[1]]
output_col_index = header_row.index(TARGET_OUTPUT_COLUMN) + 1

for row_num, flag in enumerate(not_found_flags, start=2):
    if flag:
        cell = worksheet.cell(row=row_num, column=output_col_index)
        cell.font = Font(color="FF0000")

# ------------------------------------------------------------
# Step 7: Save Final Output
# ------------------------------------------------------------
workbook.save(OUTPUT_EXCEL_PATH)

print(f"Process completed successfully.")
print(f"Output file generated at: {OUTPUT_EXCEL_PATH}")
